# Lab 2 · What extraction keeps, and how to change it

**About 25 minutes. Starts cold:** it doesn't need lab 1.

An agent writes memory one exchange at a time. You'll replay a real support conversation that way and see three things go wrong:
- **Conversation state saved as memory:** "the assistant asked…; awaiting reply".
- **A correction that doesn't replace the fact it corrects.**
- **Preferences that vanish when an old conversation is deleted.**

Then you'll fix the first one, and see why the second can't be fixed the same way.

In [ ]:
from collections import Counter

import labkit
from memory_inspector.health.checks import EXTRACTION_INSTRUCTIONS

pool, memory = labkit.connect(source="replay")
USER, FIXED, OTHER = labkit.lab_user(2), labkit.lab_user(2, "fixed"), labkit.lab_user(2, "other")
for user in (USER, FIXED, OTHER):
    labkit.reset_user(memory, user)

## 1. Replay a conversation, one exchange at a time

In this support conversation the user states a durable preference (email only) and a detail (the bucket region), then **corrects** the region two messages later. Extraction runs after every exchange, so this takes about half a minute.

In [ ]:
thread = labkit.replay(memory, USER, "support_01")
labkit.show_memories(pool, USER, flag_transient=True)

Before running the check, decide for yourself:
- **Which of these would you want an agent to know next week?**
- **Which only mattered while the conversation was open?**
- **Is the wrong region (us-east-1) still there?**

## 2. Ask the health check

In [ ]:
report = labkit.check(pool, USER)

You should see some of these:
- **`transient`:** conversation state stored as durable memory. It will be retrieved next week and take a prompt slot.
- **`superseded`:** the us-east-1 memory, still stored next to its correction. Both are retrievable, and they sit very close together, so a search for one returns the other.
- **`scope_mismatch`:** memories the extractor labelled user-scoped but stored on the thread.

Every finding comes with a fix. Here is the one for transient memories:

In [ ]:
transient = [f for f in report.findings if f.kind == "transient"]
print(transient[0].suggestion if transient else "No transient memories this run: extraction varies. Read the fix in the next cell anyway.")

## 3. Fix it: tell extraction what not to keep

`memory_extraction_custom_instructions` adds your instructions to the extraction prompt. This wording was tested: in three trials it removed every transient memory. Notice the last sentence. It asks corrections to *say what they correct*, so a reader, or the health check, can tell which fact is current.

In [ ]:
print(EXTRACTION_INSTRUCTIONS)
fixed_thread = labkit.replay(memory, FIXED, "support_01", memory_extraction_custom_instructions=EXTRACTION_INSTRUCTIONS)
labkit.show_memories(pool, FIXED, flag_transient=True)
fixed_report = labkit.check(pool, FIXED)

In [ ]:
kinds = lambda r: dict(Counter(f.kind for f in r.findings))
print("without instructions:", kinds(report))
print("with instructions:   ", kinds(fixed_report))

Transient memories should be gone. **Look for `superseded`, though: it's probably still there.**

Extraction only ever *adds* memories. When the user corrects themselves, the extractor writes the corrected fact as a new row. It never updates or removes the old one, whatever the prompt says. The instruction made the correction easier to recognise, not the stale fact go away. Cleaning that up is the agent's job, and it's what lab 3 does.

## 4. One user can't see another's memories

Search is scoped. Another user, with a different contact preference, gets their own answer:

In [ ]:
_ = labkit.replay(memory, OTHER, "support_02")
for uid in (USER, OTHER):
    results = memory.search("How does this user want to be contacted?", user_id=uid,
                            record_types=labkit.MEMORY_TYPES, max_results=3)
    print(uid)
    labkit.show_results(results, highlight="email|phone")
    assert all(r.record.user_id == uid for r in results), "a result crossed the user boundary"
print("every result belonged to the user who asked")

## 5. Delete a conversation, lose a preference

Support threads get cleaned up. Watch what happens to this user's preferences when their thread goes:

In [ ]:
before = labkit.memories(pool, USER)
print("preferences before:", [labkit.shorten(m["content"], 70) for m in before if m["type"] == "preference"])
memory.delete_thread(thread.thread_id)
after = labkit.memories(pool, USER)
print(f"{len(before)} memories → {len(after)} after deleting the thread")
print("preferences after: ", [labkit.shorten(m["content"], 70) for m in after if m["type"] == "preference"])

Every extracted memory carries the thread's id, and deleting the thread cascades to them, **including the ones the extractor itself labelled user-scoped**. That's the `scope_mismatch` finding made concrete.

The fix is to copy what should outlive the conversation to user level *before* deleting it. Do that for the `FIXED` user:

In [ ]:
keep = [m for m in labkit.memories(pool, FIXED) if m["type"] == "preference"]
for m in keep:
    memory.add_memory(m["content"], user_id=FIXED, memory_type="preference")   # no thread_id: user level
memory.delete_thread(fixed_thread.thread_id)
labkit.show_memories(pool, FIXED)

In [ ]:
memory.close()
pool.close()

## What you saw

- Extraction runs after every exchange, and it keeps **conversation state** unless told not to. Custom instructions fix that, and the check proves it.
- Extraction **appends**. A correction adds a row; it never revises the old one. No prompt fixes that. The agent has to clean up (lab 3).
- Search respects the **user boundary**.
- Extracted memories live and die **with their thread**. Copy to user level anything that should outlive the conversation.

Open the dashboard's **Memory** page to see these findings with their evidence: http://localhost:3000/memories

**Next: lab 3**, where these problems cost the agent a correct answer, and you get it back.